# **EuroSAT V2 Pipeline: Hybrid Quantum CNN (HQCNN)**

**NOTE:** Use T4 GPU Runtime on Google Colab

While the quantum circuit is small (4 qubits), the PennyLane framework uses the GPU to rapidly calculate the quantum state vectors and parameter-shift gradients.

#### **Objective**
To train a Hybrid Quantum Convolutional Neural Network (HQCNN) using **Sequential Transfer Learning**.

Instead of training 11 million classical parameters alongside 4 quantum parameters (which causes "Classical Masking"); we will:
1. Load our pre-trained, fine-tuned classical weights from Google Drive (that we saved from `02_resnet_baseline`).
2. Freeze the classical ResNet18 backbone.
3. Train **only** the classical bottleneck (dimensionality reduction) and the Quantum Layer.

This ensures the quantum circuit is forced to do the actual classification work using the high-quality features extracted by the classical backbone.

In [1]:
import os
import sys
import torch
from google.colab import drive

# Mount Drive to access our saved weights
drive.mount('/content/drive')

# Clone repo or pull latest changes
if os.path.exists('/content/QML4EO-reproduction'):
    print("Repo found! Pulling latest changes from GitHub..")
    os.chdir('/content/QML4EO-reproduction')
    !git pull
else:
    print("Cloning repo for the first time..")
    os.chdir('/content')
    !git clone https://github.com/yeshapan/QML4EO-reproduction.git
    os.chdir('/content/QML4EO-reproduction')

if '/content/QML4EO-reproduction' not in sys.path:
    sys.path.append('/content/QML4EO-reproduction')

!pip install -r requirements.txt -q

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nHardware utilized: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
Cloning repo for the first time..
Cloning into 'QML4EO-reproduction'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 159 (delta 63), reused 130 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 13.67 MiB | 11.93 MiB/s, done.
Resolving deltas: 100% (63/63), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

#### **Initialize Data Loaders**
We must use the exact same resolution ($224\times224$) that we used to fine-tune the classical model $\leftarrow$ so the spatial dimensions perfectly match the saved weights.

In [2]:
from src.utils.data_loader import get_eurosat_dataloaders

# Use the optimized loader parameters (num_workers=2, pin_memory=True)
train_loader, val_loader, classes = get_eurosat_dataloaders(
    data_dir="./data",
    batch_size=64,
    img_size=224,
    num_workers=2,
    pin_memory=True
)

100%|██████████| 94.3M/94.3M [00:00<00:00, 366MB/s]


Dataset loaded successfully! Total: 27000 | Train: 21600 | Val: 5400


#### **Initialize Hybrid Architecture & Decoupled Training**

Now we initialize the HQCNN. We pass it the path to our Google Drive weights. The architecture will automatically load them and freeze the ResNet18 layers.

We will use the `train_decoupled_hqcnn` function $→$ which applies two separate learning rates:
* standard rate for the classical bottleneck
* highly sensitive, lower rate for the quantum parameters

In [6]:
from src.models.hqcnn import HybridQCNN, train_decoupled_hqcnn
from src.baselines.cnn import set_seed
import numpy as np

# Path to the weights we generated in Notebook 2
WEIGHTS_PATH = '/content/drive/MyDrive/QML4EO-reproduction/models/finetuned_resnet18_eurosat.pth'

# Quantum Topology Settings
ENTANGLEMENT = "ring" # Options: "none", "ring", "full"
QUBITS = 4
LAYERS = 1
EPOCHS = 10
SEEDS = [42, 100, 2026]

# Containers to hold metrics for statistical analysis
all_train_losses = []
all_val_accs = []

print(f"\nStarting Multi-Seed HybridQCNN Training → Entanglement: {ENTANGLEMENT.upper()}")

for seed in SEEDS:
    print(f"\nTRAINING SEED: {seed}")
    set_seed(seed)

    # Initialize Hybrid Model with pre-trained weights
    model = HybridQCNN(
        num_classes=len(classes),
        num_qubits=QUBITS,
        num_layers=LAYERS,
        entanglement_type=ENTANGLEMENT,
        pretrained_weights_path=WEIGHTS_PATH
    )

    if seed == SEEDS[0]:
        # Only print parameters on the first seed to avoid clutter
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nTotal Model Parameters: {total_params:,}")
        print(f"Active Trainable Parameters: {trainable_params:,}\n")

    # Run the Decoupled Training Loop
    history = train_decoupled_hqcnn(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        device=device
    )

    # Save the full array of metrics for this seed
    all_train_losses.append(history['train_loss'])
    all_val_accs.append(history['val_acc'])

# Convert lists to numpy arrays to easily calculate mean and standard deviation
acc_array = np.array(all_val_accs)
loss_array = np.array(all_train_losses)

mean_acc = np.mean(acc_array, axis=0)
std_acc = np.std(acc_array, axis=0)

mean_loss = np.mean(loss_array, axis=0)
std_loss = np.std(loss_array, axis=0)

print("\nMulti-Seed Training Complete!")
print(f"Final Average Accuracy: {mean_acc[-1]:.2f}% ± {std_acc[-1]:.2f}%")


Starting Multi-Seed HybridQCNN Training → Entanglement: RING

TRAINING SEED: 42

Total Model Parameters: 11,178,618
Active Trainable Parameters: 2,106



Epoch 1/10 [Train]: 100%|██████████| 338/338 [00:58<00:00,  5.76it/s, loss=1.8963]


Epoch 1 Summary → Train Loss: 2.0508 | Val Accuracy: 60.96%


Epoch 2/10 [Train]: 100%|██████████| 338/338 [00:58<00:00,  5.75it/s, loss=1.4382]


Epoch 2 Summary → Train Loss: 1.6808 | Val Accuracy: 74.98%


...

Epoch 8 Summary → Train Loss: 1.0170 | Val Accuracy: 67.74%


Epoch 9/10 [Train]: 100%|██████████| 338/338 [00:59<00:00,  5.64it/s, loss=0.8079]


Epoch 9 Summary → Train Loss: 0.8842 | Val Accuracy: 67.57%


Epoch 10/10 [Train]: 100%|██████████| 338/338 [01:00<00:00,  5.59it/s, loss=0.7259]


Epoch 10 Summary → Train Loss: 0.8079 | Val Accuracy: 68.17%

Multi-Seed Training Complete!
Final Average Accuracy: 74.86% ± 4.97%


In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(14, 5))

# Plot Validation Accuracy with Standard Deviation Band
plt.subplot(1, 2, 1)
plt.plot(epochs_range, mean_acc, label=f'{ENTANGLEMENT} (Mean)', color='purple', marker='o', linewidth=2)
plt.fill_between(epochs_range, mean_acc - std_acc, mean_acc + std_acc, color='purple', alpha=0.2, label='± 1 Std Dev')

plt.title(f'HQCNN Validation Accuracy ({len(SEEDS)} Seeds)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='lower right')

# Plot Training Loss with Standard Deviation Band
plt.subplot(1, 2, 2)
plt.plot(epochs_range, mean_loss, label='Train Loss (Mean)', color='orange', marker='x', linewidth=2)
plt.fill_between(epochs_range, mean_loss - std_loss, mean_loss + std_loss, color='orange', alpha=0.2)

plt.title(f'HQCNN Training Loss ({len(SEEDS)} Seeds)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

#### **Empirical Observations: The Quantum Loss Landscape & Seed Variance**

This multi-seed training experiment reveals a fundamental characteristic of Quantum Machine Learning (QML): **extreme sensitivity to random weight initialization**

* Seed 42 converged rapidly to ~80% accuracy
* Subsequent seeds (like Seed 100 and Seed 2026) struggled heavily in early epochs $\implies$ resulting in a high standard deviation across the final metrics.

This behavior highlights several critical QML challenges:

* **The Dimensionality Bottleneck:** We are compressing 512 classical features into a 4-dimensional quantum state vector. If a random seed initializes the classical bottleneck layer poorly, the quantum circuit receives highly obfuscated data, delaying the learning process.

* **Barren Plateaus & The Jagged Landscape:** Classical ResNets (11M+ parameters) have smooth loss landscapes where bad initializations are easily compensated for. Our HQCNN relies on only 2,106 actively trainable parameters. If a seed initializes the quantum angles on a flat ridge of the loss landscape (a "Barren Plateau"), gradients vanish, and the model struggles to update its weights effectively (as seen in Seed 2026's early epochs).

* **Statistical Rigor in QML:** This variance mathematically proves why single-seed reporting in quantum research is unreliable. A "lucky" seed can misrepresent a model's true capability.

**Next Steps - Ablation Studies Experiments**